# 🌿 HUẤN LUYỆN BỔ SUNG DỮ LIỆU THỰC TẾ (`bo_sung_thuc_te.zip`) — MOBILENETV2 & EFFICIENTNETB0

**Mục tiêu:** Cải thiện nhận diện trên ảnh thực địa ngoài trời (ánh sáng tự nhiên, nhiều nền phức tạp, các loài cây trong sân trường) từ bộ dữ liệu đã xác minh `bo_sung_thuc_te.zip` mà **không làm suy giảm (catastrophic forgetting)** 90 loài gốc và **không làm loãng tập `val` / `test` chuẩn**.

### 📦 Cấu trúc `bo_sung_thuc_te.zip`
- `da_co_trong_90_loai/` (**6 loài đã có trong 90 lớp, 90 ảnh thực tế**): chỉ gộp vào tập `train/` và được oversample an toàn kèm augmentation ngoài trời nhẹ:
  - `Song of India (Dracaena reflexa)`
  - `Spider Plant (Chlorophytum comosum)`
  - `Asparagus Fern (Asparagus setaceus)`
  - `Moses in the Cradle (Tradescantia spathacea)`
  - `Weeping Fig (Ficus benjamina)`
  - `Anthurium (Anthurium andraeanum)`
- `loai_moi_bo_sung/` (**8 loài mới ngoài bộ 90 lớp, 129 ảnh thực tế**):
  - Mặc định `INCLUDE_NEW_SPECIES = False` để giữ nguyên **90 lớp** tương thích 100% với `ai-service/model/` và CSDL hiện tại.
  - Bật `INCLUDE_NEW_SPECIES = True` nếu muốn mở rộng thành **98 lớp** (tự động giữ mỗi loài mới tối thiểu 1 ảnh trong `val/` & `test/` để đảm bảo nhất quán `class_names.json`, và warm-start trọng số 90 lớp cũ sang đầu ra 98 lớp).

In [ ]:
# ==============================================================================
# === BƯỚC 1: KẾT NỐI GOOGLE DRIVE, CẤU HÌNH & TỰ ĐỘNG DÒ FILE TRÊN DRIVE    ===
# ==============================================================================
import os
import json
import math
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image, ImageOps
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

# --- CẤU HÌNH CHÍNH (BẠN CÓ THỂ ĐIỀU CHỈNH Ở ĐÂY) ---
RANDOM_SEED = 42
INCLUDE_NEW_SPECIES = False          # False = 90 loài chuẩn hiện tại | True = 98 loài (thêm 8 loài mới)
FAST_REPLAY_MODE = True              # True = Cách 2 (Replay Nhẹ ~15 ảnh cũ/loài, chạy ~2 phút) | False = Full 23k/30k ảnh
REPLAY_TRAIN_PER_CLASS = 15          # Số ảnh đại diện mỗi loài cũ trong tập train khi bật FAST_REPLAY_MODE
REPLAY_EVAL_PER_CLASS = 5            # Số ảnh mỗi loài cũ trong val/test khi bật FAST_REPLAY_MODE
MODELS_TO_TRAIN = ['MobileNetV2', 'EfficientNetB0']  # Có thể giữ cả 2 hoặc chỉ chọn 1 mô hình

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

# Oversampling an toàn cho ảnh thực tế CHỈ TRONG TẬP TRAIN (không đụng vào val/test gốc)
REAL_OVERSAMPLE_TARGET_KNOWN = 90    # Mục tiêu số lượt xuất hiện ảnh thực tế / epoch cho 6 loài đã có
MAX_REAL_REPEAT_KNOWN = 18           # Giới hạn lặp tối đa (tránh lặp quá mức khi chỉ có 4-5 ảnh gốc)
REAL_OVERSAMPLE_TARGET_NEW = 160     # Mục tiêu cho 8 loài mới nếu INCLUDE_NEW_SPECIES = True
MAX_NEW_REPEAT = 45

# Siêu tham số Fine-Tuning chống quên kiến thức cũ (Catastrophic Forgetting)
LR_FINETUNE = {
    'MobileNetV2': 5e-5,             # Nằm trong vùng an toàn 5e-5 .. 1e-4
    'EfficientNetB0': 8e-5,          # Nằm trong vùng an toàn 5e-5 .. 1e-4
}
LR_HEAD_WARMUP = 3e-4                # Chỉ dùng khi mở rộng sang 98 lớp hoặc train từ ImageNet
EPOCHS_HEAD_WARMUP = 20              # Tối đa; EarlyStopping (patience=5) tự dừng khi hội tụ
EPOCHS_FINETUNE = 30                 # Tối đa; EarlyStopping (patience=5) tự dừng khi hội tụ
LABEL_SMOOTHING = 0.05

# Cố định Seed tái lập
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)

PROJECT = Path('/content/drive/MyDrive/KiengLa_AI')
assert PROJECT.exists(), f'Không tìm thấy thư mục {PROJECT} trên Google Drive!'

MODE_TAG = '98_loai_mo_rong' if INCLUDE_NEW_SPECIES else '90_loai_chuan'
SAVE_ROOT = PROJECT / f'train_bo_sung_thuc_te_{MODE_TAG}_seed_{RANDOM_SEED}'
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

def find_first_existing(candidates, rglob_pattern=None):
    for p in candidates:
        if p.is_file():
            return p
    if rglob_pattern:
        found = sorted(PROJECT.rglob(rglob_pattern))
        if found:
            return found[0]
    return None

ZIP_30K_PATH = find_first_existing(
    [
        PROJECT / '01_Du_Lieu_30k_Va_Labels' / 'merged_v1' / 'merged_90_30k_chuan.zip',
        PROJECT / '01_Du_Lieu_Goc' / 'merged_90_30k_chuan.zip',
        PROJECT / 'merged_v1' / 'merged_90_30k_chuan.zip',
        PROJECT / 'merged_90_30k_chuan.zip',
        PROJECT / 'merged_v1' / 'merged_90_split.zip',
        PROJECT / '01_Du_Lieu_Goc' / 'merged_90_split.zip',
        PROJECT / 'merged_90_split.zip',
    ],
    rglob_pattern='*merged_90*.zip',
)
assert ZIP_30K_PATH is not None, f'Không tìm thấy merged_90_30k_chuan.zip hoặc merged_90_split.zip (23k ảnh) trong {PROJECT}!'

ZIP_BOSUNG_PATH = find_first_existing(
    [
        PROJECT / 'bo_sung_thuc_te.zip',
        PROJECT / '01_Du_Lieu_Goc' / 'bo_sung_thuc_te.zip',
        PROJECT / '01_Du_Lieu_30k_Va_Labels' / 'bo_sung_thuc_te.zip',
        Path('/content/bo_sung_thuc_te.zip'),
    ],
    rglob_pattern='*bo_sung_thuc_te*.zip',
)
assert ZIP_BOSUNG_PATH is not None, (
    f'Không tìm thấy bo_sung_thuc_te.zip! Hãy upload file bo_sung_thuc_te.zip lên {PROJECT}/bo_sung_thuc_te.zip'
)

PRETRAINED_CANDIDATES = {
    'MobileNetV2': [
        PROJECT / 'train_90_lop_30k_anh' / 'finetune' / 'best_model.keras',
        PROJECT / 'train_90_lop_30k_seed_999' / 'finetune' / 'best_model.keras',
        PROJECT / 'train_90_lop_30k_paper_standard' / 'best_mobilenetv2_paper.keras',
        PROJECT / 'best_model.keras',
        PROJECT / 'models' / 'best_model.keras',
    ],
    'EfficientNetB0': [
        PROJECT / 'train_90_lop_30k_paper_standard' / 'best_efficientnetb0_paper.keras',
        PROJECT / 'benchmark_so_sanh_da_mo_hinh' / 'efficientnet_b0' / 'model.keras',
        PROJECT / 'benchmark_so_sanh_da_mo_hinh' / 'efficientnet_b0' / 'best_model.keras',
        PROJECT / 'efficientnet_b0_30k.keras',
        PROJECT / 'models' / 'efficientnet_b0_30k.keras',
        PROJECT / 'benchmark_models_seed_42' / 'efficientnet_b0' / 'best_model.keras',
    ],
}

PRETRAINED_PATHS = {}
for m_name, cands in PRETRAINED_CANDIDATES.items():
    p = find_first_existing(cands)
    if p is None and m_name == 'EfficientNetB0':
        eff_cands = [
            cand for cand in sorted(PROJECT.rglob('*efficientnet*.keras'))
            if 'train_bo_sung_' not in str(cand)
        ]
        p = eff_cands[0] if eff_cands else None
    PRETRAINED_PATHS[m_name] = p

print('=' * 74)
print('🚀 THIẾT LẬP MÔI TRƯỜNG HUẤN LUYỆN BỔ SUNG THỰC TẾ')
print(f'  • TensorFlow         : {tf.__version__}')
print(f'  • GPU                : {tf.config.list_physical_devices("GPU")}')
print(f'  • INCLUDE_NEW_SPECIES: {INCLUDE_NEW_SPECIES} ({MODE_TAG})')
print(f'  • MODELS_TO_TRAIN    : {MODELS_TO_TRAIN}')
print(f'  • merged_90_30k      : {ZIP_30K_PATH}')
print(f'  • bo_sung_thuc_te    : {ZIP_BOSUNG_PATH}')
for m_name, p in PRETRAINED_PATHS.items():
    print(f'  • Pretrained {m_name:<15}: {p if p else "Không có sẵn (sẽ fallback từ ImageNet)"}')
print(f'  • Thư mục lưu Drive  : {SAVE_ROOT}')
print('=' * 74)

In [ ]:
# ==============================================================================
# === BƯỚC 2: GIẢI NÉN & TẠO TẬP TRAIN/VAL/TEST KHÔNG LÀM LOÃNG VAL/TEST GỐC ===
# ==============================================================================
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

RAW_30K_DIR = Path('/content/raw_30k_extracted')
RAW_BOSUNG_DIR = Path('/content/raw_bo_sung_extracted')
DATA_SPLIT_DIR = Path(f'/content/data_bo_sung_split_{MODE_TAG}_seed_{RANDOM_SEED}')

# 1. Giải nén tập gốc 30k (90 loài)
if not RAW_30K_DIR.exists() or len(list(RAW_30K_DIR.glob('*'))) == 0:
    RAW_30K_DIR.mkdir(parents=True, exist_ok=True)
    print(f'📦 Đang giải nén {ZIP_30K_PATH.name}...')
    !unzip -q "{ZIP_30K_PATH}" -d /content/raw_30k_extracted
    print('✅ Đã giải nén merged_90_30k_chuan.zip')

# 2. Giải nén bo_sung_thuc_te.zip
if RAW_BOSUNG_DIR.exists():
    shutil.rmtree(RAW_BOSUNG_DIR)
RAW_BOSUNG_DIR.mkdir(parents=True, exist_ok=True)
print(f'📦 Đang giải nén {ZIP_BOSUNG_PATH.name}...')
!unzip -q "{ZIP_BOSUNG_PATH}" -d /content/raw_bo_sung_extracted

def locate_subdir(root_dir: Path, target_name: str) -> Path:
    direct = root_dir / target_name
    if direct.is_dir():
        return direct
    matches = [p for p in root_dir.rglob(target_name) if p.is_dir()]
    assert matches, f'Không tìm thấy thư mục {target_name} bên trong {root_dir}!'
    return matches[0]

KNOWN_SUPP_DIR = locate_subdir(RAW_BOSUNG_DIR, 'da_co_trong_90_loai')
NEW_SUPP_DIR = locate_subdir(RAW_BOSUNG_DIR, 'loai_moi_bo_sung')

def collect_class_images(base_dir: Path):
    class_map = {}
    for root, _, files in os.walk(str(base_dir)):
        folder_name = os.path.basename(root)
        if folder_name.startswith('.'):
            continue
        valid_f = sorted(
            [Path(root) / f for f in files if Path(f).suffix.lower() in EXTENSIONS],
            key=lambda p: p.name,
        )
        if valid_f and Path(root) != base_dir:
            class_map.setdefault(folder_name, []).extend(valid_f)
    return class_map

base_class_images = collect_class_images(RAW_30K_DIR)
base_class_folders = sorted(base_class_images.keys())
assert len(base_class_folders) == 90, f'Tập gốc phải đúng 90 loài, phát hiện {len(base_class_folders)}!'

known_supp_images = collect_class_images(KNOWN_SUPP_DIR)
new_supp_images = collect_class_images(NEW_SUPP_DIR)

# Kiểm tra khớp 100% tên nhãn gốc của 6 loài bổ sung
for cls_name in known_supp_images:
    assert cls_name in base_class_images, f'Nhãn {cls_name!r} không khớp tập 90 loài gốc!'

def link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

flag_split = DATA_SPLIT_DIR / f'_split_{MODE_TAG}_ok'
if not flag_split.exists():
    if DATA_SPLIT_DIR.exists():
        shutil.rmtree(DATA_SPLIT_DIR)

    rng = random.Random(RANDOM_SEED)
    split_rows = []

    # A. Chia tập gốc 90 loài theo tỷ lệ 80-10-10 (val & test giữ nguyên 100% tinh khiết)
    for cls_name in base_class_folders:
        imgs = sorted(list(set(base_class_images[cls_name])), key=lambda p: p.name)
        rng.shuffle(imgs)
        n = len(imgs)
        n_tr = int(n * 0.80)
        n_va = int(n * 0.10)
        if n_va == 0 and n >= 3:
            n_va = 1
        tr_list = imgs[:n_tr]
        va_list = imgs[n_tr:n_tr + n_va]
        te_list = imgs[n_tr + n_va:]
        if FAST_REPLAY_MODE:
            tr_list = tr_list[:REPLAY_TRAIN_PER_CLASS]
            va_list = va_list[:REPLAY_EVAL_PER_CLASS]
            te_list = te_list[:REPLAY_EVAL_PER_CLASS]
        splits = {
            'train': tr_list,
            'val': va_list,
            'test': te_list,
        }
        for s_name, paths in splits.items():
            for p_src in paths:
                link_or_copy(p_src, DATA_SPLIT_DIR / s_name / cls_name / p_src.name)

        # B. Gộp ảnh thực tế (da_co_trong_90_loai) CHỈ VÀO train/, có oversample an toàn
        real_paths = sorted(known_supp_images.get(cls_name, []), key=lambda p: p.name)
        real_repeat = 0
        real_effective = 0
        if real_paths:
            real_repeat = min(MAX_REAL_REPEAT_KNOWN, max(1, REAL_OVERSAMPLE_TARGET_KNOWN // len(real_paths)))
            for rep in range(real_repeat):
                for idx_r, p_real in enumerate(real_paths):
                    dst_name = f'bo_sung_r{rep:02d}_{idx_r:03d}_{p_real.name}'
                    link_or_copy(p_real, DATA_SPLIT_DIR / 'train' / cls_name / dst_name)
            real_effective = len(real_paths) * real_repeat

        split_rows.append({
            'class_name': cls_name,
            'source_type': 'known_supplemented' if real_paths else 'known_base_only',
            'base_train': len(splits['train']),
            'real_orig': len(real_paths),
            'real_repeat': real_repeat,
            'real_train_added': real_effective,
            'total_train': len(splits['train']) + real_effective,
            'val': len(splits['val']),
            'test': len(splits['test']),
        })

    # C. Nếu INCLUDE_NEW_SPECIES = True -> thêm 8 loài mới (giữ >=1 ảnh val & >=1 ảnh test cho chuẩn Keras)
    if INCLUDE_NEW_SPECIES:
        for new_cls in sorted(new_supp_images.keys()):
            imgs = sorted(new_supp_images[new_cls], key=lambda p: p.name)
            rng.shuffle(imgs)
            n = len(imgs)
            assert n >= 3, f'Loài mới {new_cls} cần tối thiểu 3 ảnh để chia train/val/test, hiện có {n}'
            n_va = max(1, int(round(n * 0.10)))
            n_te = max(1, int(round(n * 0.10)))
            if n_va + n_te >= n:
                n_va, n_te = 1, 1
            tr_imgs = imgs[:n - n_va - n_te]
            va_imgs = imgs[n - n_va - n_te:n - n_te]
            te_imgs = imgs[n - n_te:]

            for p_src in va_imgs:
                link_or_copy(p_src, DATA_SPLIT_DIR / 'val' / new_cls / p_src.name)
            for p_src in te_imgs:
                link_or_copy(p_src, DATA_SPLIT_DIR / 'test' / new_cls / p_src.name)

            new_repeat = min(MAX_NEW_REPEAT, max(1, REAL_OVERSAMPLE_TARGET_NEW // len(tr_imgs)))
            for rep in range(new_repeat):
                for idx_n, p_src in enumerate(tr_imgs):
                    dst_name = f'new_r{rep:02d}_{idx_n:03d}_{p_src.name}'
                    link_or_copy(p_src, DATA_SPLIT_DIR / 'train' / new_cls / dst_name)

            split_rows.append({
                'class_name': new_cls,
                'source_type': 'new_species',
                'base_train': 0,
                'real_orig': len(tr_imgs),
                'real_repeat': new_repeat,
                'real_train_added': len(tr_imgs) * new_repeat,
                'total_train': len(tr_imgs) * new_repeat,
                'val': len(va_imgs),
                'test': len(te_imgs),
            })

    split_df = pd.DataFrame(split_rows)
    split_df.to_csv(SAVE_ROOT / 'dataset_split_summary.csv', index=False, encoding='utf-8-sig')
    flag_split.touch()
else:
    split_df = pd.read_csv(SAVE_ROOT / 'dataset_split_summary.csv')

print('=' * 74)
print(f'📊 TỔNG KẾT TẬP DỮ LIỆU ({len(split_df)} LOÀI):')
print(f'  • Train tổng (sau oversample thực tế): {int(split_df["total_train"].sum()):,} ảnh')
print(f'  • Val tổng (tinh khiết, không loãng)  : {int(split_df["val"].sum()):,} ảnh')
print(f'  • Test tổng (tinh khiết, độc lập)     : {int(split_df["test"].sum()):,} ảnh')
print('\n📌 Chi tiết các lớp có bổ sung ảnh thực tế:')
supp_view = split_df[split_df['source_type'] != 'known_base_only']
display(supp_view[['class_name', 'source_type', 'base_train', 'real_orig', 'real_repeat', 'total_train', 'val', 'test']])

In [ ]:
# ==============================================================================
# === BƯỚC 3: TẠO TF.DATA PIPELINE, MILD OUTDOOR AUGMENTATION & CLASS WEIGHTS ===
# ==============================================================================
def load_split_ds(split_name: str, shuffle: bool):
    return keras.utils.image_dataset_from_directory(
        str(DATA_SPLIT_DIR / split_name),
        image_size=IMAGE_SIZE,
        interpolation='bilinear',
        crop_to_aspect_ratio=True,
        batch_size=BATCH_SIZE,
        label_mode='categorical',
        shuffle=shuffle,
        seed=RANDOM_SEED,
    )

train_ds_raw = load_split_ds('train', shuffle=True)
val_ds_raw = load_split_ds('val', shuffle=False)
test_ds_raw = load_split_ds('test', shuffle=False)

class_names = list(train_ds_raw.class_names)
NUM_CLASSES = len(class_names)
EXPECTED_CLASSES = 98 if INCLUDE_NEW_SPECIES else 90
assert NUM_CLASSES == EXPECTED_CLASSES, f'Kỳ vọng {EXPECTED_CLASSES} lớp, nhận được {NUM_CLASSES}!'
assert list(val_ds_raw.class_names) == class_names, 'Thứ tự lớp val không khớp train!'
assert list(test_ds_raw.class_names) == class_names, 'Thứ tự lớp test không khớp train!'

# Lưu class_names.json ngay vào Google Drive
(SAVE_ROOT / 'class_names.json').write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
if INCLUDE_NEW_SPECIES:
    labels_template = [
        {
            'class_id': idx,
            'original_label': name,
            'is_new_species': name in new_supp_images,
        }
        for idx, name in enumerate(class_names)
    ]
    (SAVE_ROOT / 'labels_98_template.json').write_text(
        json.dumps(labels_template, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

# Damped Class Weights (Cui et al., CVPR 2019) giúp ổn định 84 lớp không bổ sung và nâng nhẹ lớp ít ảnh
y_train_indices = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in train_ds_raw], axis=0)
counts = np.bincount(y_train_indices, minlength=NUM_CLASSES).astype(np.float32)
mean_count = np.mean(counts)
damped_weights = np.clip((mean_count / np.maximum(counts, 1.0)) ** 0.5, 0.65, 2.20)
class_weight_dict = {i: float(w) for i, w in enumerate(damped_weights)}

# Mild Outdoor Photometric Augmentation CHỈ áp dụng lên train_ds (giữ pixel hợp lệ [0, 255])
# Lưu ý: các model gốc đã có sẵn lớp RandomFlip/Rotation/Zoom bên trong đồ thị Keras,
# nên ở đây chỉ bổ sung biến thiên ánh nắng/bóng râm nhẹ ngoài trời để không làm méo lá.
def mild_outdoor_augment(images, labels):
    images = tf.cast(images, tf.float32)
    images = tf.image.random_brightness(images, max_delta=20.0)
    images = tf.clip_by_value(images, 0.0, 255.0)
    images = tf.image.random_contrast(images, lower=0.86, upper=1.14)
    images = tf.clip_by_value(images, 0.0, 255.0)
    images = tf.image.random_saturation(images, lower=0.88, upper=1.15)
    images = tf.clip_by_value(images, 0.0, 255.0)
    return images, labels

train_ds = train_ds_raw.map(mild_outdoor_augment, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds_raw.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds_raw.prefetch(tf.data.AUTOTUNE)

KNOWN_SUPPLEMENTED_SET = set(known_supp_images.keys())
NEW_SPECIES_SET = set(new_supp_images.keys()) if INCLUDE_NEW_SPECIES else set()

print(f'✅ Data Pipeline sẵn sàng: {NUM_CLASSES} lớp | {len(y_train_indices):,} ảnh train (đã cân bằng)')
print(f'📁 Đã lưu class_names.json tại: {SAVE_ROOT / "class_names.json"}')

In [ ]:
# ==============================================================================
# === BƯỚC 4: CÁC HÀM ĐÁNH GIÁ & CHUẨN BỊ MÔ HÌNH MOBILENETV2 / EFFICIENTNETB0 ===
# ==============================================================================
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Nạp model .keras an toàn (tự loại bỏ quantization_config trong config.json, KHÔNG monkey-patch Dense)
def _strip_quant_cfg(obj):
    if isinstance(obj, dict):
        obj.pop('quantization_config', None)
        for v in obj.values():
            _strip_quant_cfg(v)
    elif isinstance(obj, list):
        for v in obj:
            _strip_quant_cfg(v)

def compat_load_model(model_path: Path):
    import tempfile
    try:
        return keras.models.load_model(str(model_path), compile=False)
    except Exception:
        with tempfile.TemporaryDirectory() as tmpd:
            tmp_keras = Path(tmpd) / model_path.name
            with zipfile.ZipFile(model_path, 'r') as zin, zipfile.ZipFile(tmp_keras, 'w') as zout:
                for item in zin.infolist():
                    data = zin.read(item.filename)
                    if item.filename == 'config.json':
                        cfg = json.loads(data.decode('utf-8'))
                        _strip_quant_cfg(cfg)
                        data = json.dumps(cfg).encode('utf-8')
                    zout.writestr(item, data)
            return keras.models.load_model(str(tmp_keras), compile=False)

def evaluate_dataset(eval_model, dataset, ds_class_names, model_class_names):
    """
    Đánh giá sạch trên test_ds, tự động ánh xạ đúng chỉ số lớp kể cả khi
    so sánh Before (90 lớp) trên test_ds của chế độ 98 lớp (INCLUDE_NEW_SPECIES=True).
    """
    ds_to_model = {
        ds_idx: model_class_names.index(name)
        for ds_idx, name in enumerate(ds_class_names)
        if name in model_class_names
    }
    y_true_all, scores_all = [], []
    for imgs, lbls in dataset:
        preds = eval_model(imgs, training=False).numpy()
        raw_y = np.argmax(lbls.numpy(), axis=1)
        keep_mask = np.array([int(idx) in ds_to_model for idx in raw_y], dtype=bool)
        if not np.any(keep_mask):
            continue
        mapped_y = np.array([ds_to_model[int(idx)] for idx in raw_y[keep_mask]], dtype=np.int64)
        y_true_all.append(mapped_y)
        scores_all.append(preds[keep_mask])

    y_true = np.concatenate(y_true_all)
    scores = np.concatenate(scores_all)
    y_pred = np.argmax(scores, axis=1)

    top1 = float(accuracy_score(y_true, y_pred)) if len(y_true) else 0.0
    top3 = float(np.mean(np.any(np.argsort(scores, axis=1)[:, -3:] == y_true[:, None], axis=1))) if len(y_true) else 0.0
    wf1 = float(f1_score(y_true, y_pred, average='weighted', zero_division=0)) if len(y_true) else 0.0

    per_class = {}
    for cls_idx, cls_name in enumerate(model_class_names):
        cls_mask = (y_true == cls_idx)
        if np.any(cls_mask):
            per_class[cls_name] = {
                'count': int(np.sum(cls_mask)),
                'acc': float(np.mean(y_pred[cls_mask] == cls_idx)),
            }

    supp_indices = [model_class_names.index(c) for c in KNOWN_SUPPLEMENTED_SET if c in model_class_names]
    supp_mask = np.isin(y_true, supp_indices)
    supp_acc = float(np.mean(y_pred[supp_mask] == y_true[supp_mask])) if np.any(supp_mask) else None

    untouched_names = [c for c in base_class_folders if c not in KNOWN_SUPPLEMENTED_SET and c in model_class_names]
    untouched_indices = [model_class_names.index(c) for c in untouched_names]
    untouched_mask = np.isin(y_true, untouched_indices)
    untouched_acc = float(np.mean(y_pred[untouched_mask] == y_true[untouched_mask])) if np.any(untouched_mask) else None

    new_acc = None
    if NEW_SPECIES_SET:
        new_indices = [model_class_names.index(c) for c in NEW_SPECIES_SET if c in model_class_names]
        new_mask = np.isin(y_true, new_indices)
        new_acc = float(np.mean(y_pred[new_mask] == y_true[new_mask])) if np.any(new_mask) else None

    return {
        'samples': int(len(y_true)),
        'top1_acc': top1,
        'top3_acc': top3,
        'weighted_f1': wf1,
        'known_6_supp_acc': supp_acc,
        'known_84_untouched_acc': untouched_acc,
        'new_8_species_acc': new_acc,
        'per_class': per_class,
        'y_true': y_true,
        'y_pred': y_pred,
        'scores': scores,
    }

def evaluate_real_world_probe(eval_model, folder_dir: Path, model_class_names):
    """Kiểm tra khả năng nhận diện trên chính bộ ảnh thực tế ngoài trời (tham khảo)."""
    correct, total = 0, 0
    details = {}
    for cls_dir in sorted(folder_dir.iterdir()):
        if not cls_dir.is_dir() or cls_dir.name not in model_class_names:
            continue
        target_idx = model_class_names.index(cls_dir.name)
        imgs = sorted([p for p in cls_dir.iterdir() if p.suffix.lower() in EXTENSIONS])
        if not imgs:
            continue
        batch = []
        for p in imgs:
            with Image.open(p) as im:
                im_rgb = ImageOps.exif_transpose(im).convert('RGB')
                im_fit = ImageOps.fit(im_rgb, IMAGE_SIZE, method=Image.Resampling.BILINEAR)
                batch.append(np.array(im_fit, dtype=np.float32))
        arr = np.stack(batch, axis=0)
        preds = eval_model(arr, training=False).numpy()
        hits = int(np.sum(np.argmax(preds, axis=1) == target_idx))
        correct += hits
        total += len(imgs)
        details[cls_dir.name] = {'hits': hits, 'total': len(imgs), 'acc': hits / len(imgs)}
    return {'acc': (correct / total if total else 0.0), 'correct': correct, 'total': total, 'details': details}

def find_backbone_submodel(model: keras.Model, backbone_name: str):
    key = backbone_name.lower()
    for lyr in model.layers:
        if isinstance(lyr, keras.Model) and (key in lyr.name.lower() or 'mobilenet' in lyr.name.lower() or 'efficientnet' in lyr.name.lower()):
            return lyr
    return None

def build_fresh_model(backbone_name: str, num_classes: int):
    inputs = keras.Input(shape=(224, 224, 3), name='image_input')
    x = layers.RandomFlip('horizontal', seed=RANDOM_SEED)(inputs)
    x = layers.RandomRotation(0.08, seed=RANDOM_SEED)(x)
    x = layers.RandomZoom(0.10, seed=RANDOM_SEED)(x)
    if backbone_name == 'MobileNetV2':
        x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name='rescaling')(x)
        base_model = keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    elif backbone_name == 'EfficientNetB0':
        base_model = keras.applications.EfficientNetB0(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    else:
        raise ValueError(f'Không hỗ trợ backbone: {backbone_name}')
    base_model.trainable = False
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.30, seed=RANDOM_SEED, name='head_drop')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name=f'pred_{num_classes}')(x)
    return keras.Model(inputs, outputs, name=f'{backbone_name}_bo_sung_{num_classes}'), base_model

def expand_90_to_98_head(base_90_model: keras.Model, new_class_names: list, old_class_names_90: list):
    """Mở rộng đầu ra Dense 90 -> 98 lớp, giữ nguyên trọng số 90 lớp cũ theo đúng ánh xạ tên lớp."""
    old_dense = base_90_model.layers[-1]
    old_w, old_b = old_dense.get_weights()
    assert old_w.shape[1] == len(old_class_names_90), (
        f'Số lớp của model cũ ({old_w.shape[1]}) khác {len(old_class_names_90)}!'
    )
    penultimate_out = base_90_model.layers[-2].output

    new_dense_layer = layers.Dense(len(new_class_names), activation='softmax', name=f'pred_{len(new_class_names)}')
    new_out = new_dense_layer(penultimate_out)
    expanded_model = keras.Model(inputs=base_90_model.inputs, outputs=new_out, name=f'{base_90_model.name}_expanded_{len(new_class_names)}')

    # Warm-start: copy cột trọng số tương ứng với 90 lớp cũ
    new_w, new_b = new_dense_layer.get_weights()
    for old_idx, lbl in enumerate(old_class_names_90):
        if lbl in new_class_names:
            new_idx = new_class_names.index(lbl)
            new_w[:, new_idx] = old_w[:, old_idx]
            new_b[new_idx] = old_b[old_idx]
    new_dense_layer.set_weights([new_w, new_b])
    return expanded_model

def set_finetune_trainable(model: keras.Model, backbone_name: str, unfreeze_ratio: float = 0.35):
    base_model = find_backbone_submodel(model, backbone_name)
    assert base_model is not None, f'Không tìm thấy backbone {backbone_name} trong model!'
    base_model.trainable = True
    n_layers = len(base_model.layers)
    n_unfreeze = max(25, int(n_layers * unfreeze_ratio))
    for lyr in base_model.layers[:-n_unfreeze]:
        lyr.trainable = False
    for lyr in base_model.layers[-n_unfreeze:]:
        # Luôn khóa BatchNormalization để giữ nguyên thống kê phân bố ImageNet/90-loài cũ
        lyr.trainable = not isinstance(lyr, layers.BatchNormalization)
    for lyr in model.layers:
        if lyr is not base_model and not isinstance(lyr, layers.BatchNormalization):
            lyr.trainable = True
    return n_unfreeze, n_layers

print('✅ Đã nạp đầy đủ hàm đánh giá và fine-tuning an toàn.')

In [ ]:
# ==============================================================================
# === BƯỚC 5: HUẤN LUYỆN & ĐÁNH GIÁ BEFORE vs AFTER (MOBILENETV2 / EFFNETB0) ===
# ==============================================================================
comparison_rows = []
per_class_before_after = {}

for backbone_name in MODELS_TO_TRAIN:
    print('\n' + '=' * 78)
    print(f'🔬 BẮT ĐẦU QUY TRÌNH CHO MÔ HÌNH: {backbone_name}')
    print('=' * 78)

    model_dir = SAVE_ROOT / backbone_name.lower()
    model_dir.mkdir(parents=True, exist_ok=True)
    ckpt_tmp_path = model_dir / f'_ckpt_{backbone_name.lower()}.keras'
    save_model_path = model_dir / f'best_{backbone_name.lower()}_bo_sung_{MODE_TAG}.keras'

    pretrained_path = PRETRAINED_PATHS.get(backbone_name)
    loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
    metrics_list = [
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc'),
    ]

    # 1. Nạp hoặc khởi tạo mô hình & đánh giá BEFORE
    if pretrained_path is not None and pretrained_path.is_file():
        print(f'📥 Nạp mô hình tiền huấn luyện (Before): {pretrained_path}')
        before_90_model = compat_load_model(pretrained_path)

        before_test_res = evaluate_dataset(
            before_90_model,
            test_ds,
            ds_class_names=class_names,
            model_class_names=base_class_folders,
        )
        before_real_probe = evaluate_real_world_probe(
            before_90_model,
            KNOWN_SUPP_DIR,
            base_class_folders,
        )

        if not INCLUDE_NEW_SPECIES:
            model = compat_load_model(pretrained_path)
            need_head_warmup = False
        else:
            model = expand_90_to_98_head(before_90_model, class_names, base_class_folders)
            need_head_warmup = True
    else:
        print(f'⚠️ Không tìm thấy checkpoint 90 loài cho {backbone_name}; khởi tạo từ ImageNet...')
        model, _ = build_fresh_model(backbone_name, NUM_CLASSES)
        before_test_res = evaluate_dataset(
            model, test_ds, ds_class_names=class_names, model_class_names=class_names
        )
        before_real_probe = evaluate_real_world_probe(model, KNOWN_SUPP_DIR, class_names)
        need_head_warmup = True

    print(
        f'📊 [BEFORE] Test Top-1 = {before_test_res["top1_acc"]*100:.2f}% | '
        f'Top-3 = {before_test_res["top3_acc"]*100:.2f}% | '
        f'6 lớp bổ sung (Test) = {before_test_res["known_6_supp_acc"]*100:.2f}% | '
        f'84 lớp khác (Test) = {before_test_res["known_84_untouched_acc"]*100:.2f}% | '
        f'Ảnh thực tế 90 ảnh = {before_real_probe["acc"]*100:.2f}%'
    )

    # 2. Giai đoạn Warm-up Head (nếu mở rộng lên 98 lớp hoặc train từ ImageNet)
    if need_head_warmup:
        base_sub = find_backbone_submodel(model, backbone_name)
        if base_sub is not None:
            base_sub.trainable = False
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=LR_HEAD_WARMUP, clipnorm=1.0),
            loss=loss_fn,
            metrics=metrics_list,
        )
        cb_head = [
            keras.callbacks.ModelCheckpoint(
                str(ckpt_tmp_path), monitor='val_accuracy', mode='max', save_best_only=True, verbose=1
            ),
            keras.callbacks.EarlyStopping(
                monitor='val_loss', mode='min', patience=5, restore_best_weights=True, verbose=1
            ),
        ]
        print(
            f'🔥 Giai đoạn Head Warm-up (tối đa {EPOCHS_HEAD_WARMUP} epochs, '
            f'EarlyStopping patience=5, LR={LR_HEAD_WARMUP})...'
        )
        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS_HEAD_WARMUP,
            class_weight=class_weight_dict,
            callbacks=cb_head,
        )
        if ckpt_tmp_path.is_file():
            model = compat_load_model(ckpt_tmp_path)

    # 3. Giai đoạn Fine-Tuning an toàn với LR = 5e-5 .. 8e-5 & BatchNorm đóng băng
    n_unfrozen, n_total_backbone = set_finetune_trainable(model, backbone_name, unfreeze_ratio=0.35)
    lr_ft = LR_FINETUNE[backbone_name]
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_ft, clipnorm=1.0),
        loss=loss_fn,
        metrics=metrics_list,
    )
    if ckpt_tmp_path.exists():
        ckpt_tmp_path.unlink()

    cb_fine = [
        keras.callbacks.ModelCheckpoint(
            str(ckpt_tmp_path), monitor='val_accuracy', mode='max', save_best_only=True, verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_loss', mode='min', patience=5, restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1
        ),
        keras.callbacks.CSVLogger(str(model_dir / f'history_{backbone_name.lower()}.csv')),
    ]

    print(
        f'🔧 Fine-Tuning {n_unfrozen}/{n_total_backbone} lớp cuối của {backbone_name} '
        f'(BatchNorm đóng băng, LR={lr_ft}, tối đa {EPOCHS_FINETUNE} epochs, EarlyStopping patience=5)...'
    )
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINETUNE,
        class_weight=class_weight_dict,
        callbacks=cb_fine,
    )

    # Nạp checkpoint tốt nhất từ ModelCheckpoint và xuất bản sạch không kèm Optimizer
    best_trained = compat_load_model(ckpt_tmp_path) if ckpt_tmp_path.is_file() else model
    best_trained.save(str(save_model_path), include_optimizer=False)
    if ckpt_tmp_path.exists():
        ckpt_tmp_path.unlink()

    (model_dir / 'class_names.json').write_text(
        json.dumps(class_names, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    print(f'💾 Đã lưu mô hình mới tại: {save_model_path} ({save_model_path.stat().st_size / (1024*1024):.2f} MB)')

    # 4. Đánh giá AFTER trên tập Test độc lập & ảnh thực tế
    after_model = compat_load_model(save_model_path)
    after_test_res = evaluate_dataset(
        after_model, test_ds, ds_class_names=class_names, model_class_names=class_names
    )
    after_real_probe = evaluate_real_world_probe(after_model, KNOWN_SUPP_DIR, class_names)

    print(
        f'🏆 [AFTER ] Test Top-1 = {after_test_res["top1_acc"]*100:.2f}% | '
        f'Top-3 = {after_test_res["top3_acc"]*100:.2f}% | '
        f'6 lớp bổ sung (Test) = {after_test_res["known_6_supp_acc"]*100:.2f}% | '
        f'84 lớp khác (Test) = {after_test_res["known_84_untouched_acc"]*100:.2f}% | '
        f'Ảnh thực tế 90 ảnh = {after_real_probe["acc"]*100:.2f}%'
    )
    if INCLUDE_NEW_SPECIES and after_test_res['new_8_species_acc'] is not None:
        new_probe = evaluate_real_world_probe(after_model, NEW_SUPP_DIR, class_names)
        print(
            f'🌱 [8 LOÀI MỚI] Test held-out = {after_test_res["new_8_species_acc"]*100:.2f}% | '
            f'Toàn bộ 129 ảnh loài mới = {new_probe["acc"]*100:.2f}%'
        )

    rep = classification_report(
        after_test_res['y_true'],
        after_test_res['y_pred'],
        labels=np.arange(NUM_CLASSES),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(rep).transpose().to_csv(model_dir / 'classification_report_after.csv', encoding='utf-8-sig')

    comparison_rows.append({
        'model': backbone_name,
        'num_classes': NUM_CLASSES,
        'before_test_top1': round(before_test_res['top1_acc'] * 100, 2),
        'after_test_top1': round(after_test_res['top1_acc'] * 100, 2),
        'delta_test_top1': round((after_test_res['top1_acc'] - before_test_res['top1_acc']) * 100, 2),
        'before_test_top3': round(before_test_res['top3_acc'] * 100, 2),
        'after_test_top3': round(after_test_res['top3_acc'] * 100, 2),
        'before_6_supp_test': round((before_test_res['known_6_supp_acc'] or 0) * 100, 2),
        'after_6_supp_test': round((after_test_res['known_6_supp_acc'] or 0) * 100, 2),
        'before_84_other_test': round((before_test_res['known_84_untouched_acc'] or 0) * 100, 2),
        'after_84_other_test': round((after_test_res['known_84_untouched_acc'] or 0) * 100, 2),
        'before_real_90_acc': round(before_real_probe['acc'] * 100, 2),
        'after_real_90_acc': round(after_real_probe['acc'] * 100, 2),
        'saved_keras_path': str(save_model_path),
    })

    per_cls_rows = []
    for lbl in sorted(KNOWN_SUPPLEMENTED_SET):
        b_test = before_test_res['per_class'].get(lbl, {}).get('acc', 0.0)
        a_test = after_test_res['per_class'].get(lbl, {}).get('acc', 0.0)
        b_real = before_real_probe['details'].get(lbl, {}).get('acc', 0.0)
        a_real = after_real_probe['details'].get(lbl, {}).get('acc', 0.0)
        per_cls_rows.append({
            'class_name': lbl,
            'test_before_%': round(b_test * 100, 2),
            'test_after_%': round(a_test * 100, 2),
            'real_before_%': round(b_real * 100, 2),
            'real_after_%': round(a_real * 100, 2),
        })
    per_class_before_after[backbone_name] = pd.DataFrame(per_cls_rows)
    per_class_before_after[backbone_name].to_csv(
        model_dir / 'per_class_6_supplemented_before_after.csv', index=False, encoding='utf-8-sig'
    )

In [ ]:
# ==============================================================================
# === BƯỚC 6: TỔNG HỢP BÁO CÁO BEFORE vs AFTER VÀ LƯU KẾT QUẢ RA GOOGLE DRIVE ===
# ==============================================================================
df_summary = pd.DataFrame(comparison_rows)
df_summary.to_csv(SAVE_ROOT / 'before_after_comparison.csv', index=False, encoding='utf-8-sig')
(SAVE_ROOT / 'before_after_comparison.json').write_text(
    json.dumps(comparison_rows, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('=' * 78)
print('📊 BẢNG SO SÁNH BEFORE vs AFTER TỔNG HỢP:')
print('=' * 78)
display(df_summary)

for m_name, df_cls in per_class_before_after.items():
    print(f'\n🌿 Chi tiết 6 loài bổ sung thực tế ({m_name}):')
    display(df_cls)

print(f'\n✅ Hoàn tất! Toàn bộ model (.keras) và class_names.json đã lưu trong: {SAVE_ROOT}')